> **Gabarito.** Abra só depois de tentar resolver a versão do aluno.

# Aula 3 – Validar antes de processar

**Objetivo:** escrever código que falha cedo e diz ao usuário o que corrigir.

## Preparação
Rode esta célula primeiro. Ela importa as funções das aulas anteriores.

In [ ]:
import sys
from pathlib import Path
# Encontra a pasta aulas/ (onde está curso_utils.py), rodando de aulas/ ou de aulas/solucoes/
_aqui = Path.cwd()
_aulas = _aqui if (_aqui / 'curso_utils.py').exists() else _aqui.parent
sys.path.insert(0, str(_aulas))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from curso_utils import carregar_dominio, NIR_FILE

## Exercício 1 – Função `validar`
Regras:
- **Crítico** (gera `ValueError`): nenhuma coluna espectral; só uma coluna lida (delimitador errado); valores negativos.
- **Aviso** (só imprime): mais de 1% de valores faltantes.

In [ ]:
def validar(X, espec, nome):
    criticos, avisos = [], []
    if len(espec) == 0:
        criticos.append('Nenhuma coluna espectral encontrada.')
    if X.ndim != 2 or X.shape[1] <= 1:
        criticos.append('Só uma coluna lida: verifique o delimitador (; ou ,).')
    n_neg = int(np.sum(X[np.isfinite(X)] < 0))
    if n_neg:
        criticos.append(f'{n_neg} valores negativos.')
    frac_nan = float(np.mean(~np.isfinite(X)))
    if frac_nan > 0.01:
        avisos.append(f'{frac_nan:.1%} de valores faltantes.')
    for a in avisos:
        print(f'[{nome}] AVISO: {a}')
    if criticos:
        raise ValueError(f'[{nome}] ' + ' | '.join(criticos))
    print(f'[{nome}] validação OK: {X.shape}')

## Exercício 2 – Testar com dados reais e com dados "quebrados"

In [ ]:
meta, X, wl = carregar_dominio(NIR_FILE)
validar(X, list(wl), 'NIR')

# Dados quebrados de propósito
X_ruim = X[:50].copy()
X_ruim[0, 10] = -0.3
X_ruim[1:5, :] = np.nan
try:
    validar(X_ruim, list(wl), 'NIR_ruim')
except ValueError as e:
    print('Erro capturado ->', e)

## Exercício 3 – Detectar vírgula decimal
Escreva `tem_virgula_decimal(valores)` que devolve `True` se algum texto da lista parece `0,123`. Use `re.match(r"^[-+]?\d+,\d+$", s)`.

In [ ]:
import re

def tem_virgula_decimal(valores):
    return any(re.match(r'^[-+]?\d+,\d+$', str(s).strip()) for s in valores)

print(tem_virgula_decimal(['0.12', '0.33']))
print(tem_virgula_decimal(['0,12', '0.33']))

## Checkpoint

Em grupo: valores negativos devem ser críticos ou apenas aviso? Justifique a regra e aplique-a de forma consistente em todo o código.